# V14 thesis experiment pipeline: reader guide

**Companion to the thesis appendix**

This notebook explains the structure, configuration, execution, and
audit trail of `V14_Thesis_Pipeline.py`. It is a methodological guide;
it does not reproduce the confidential study dataset and does not
report thesis results.

The pipeline preserves six core model families: L-SLR, Augmented SLR,
Random Forest, XGBoost, CatBoost, and TabPFN. Its principal purpose is
to execute scenario × sample-size × iteration experiments from JSON
definitions while retaining the numerical evidence required for
independent validation.

## Goal

After reading this notebook, a reader should be able to:

1. identify the main stages of the experiment;
2. understand which decisions are supplied through JSON;
3. distinguish tuned-reference and non-tuned-reference runtime budgets;
4. prepare a configuration for a new dataset;
5. run, resume, validate, and regenerate figures; and
6. locate the artifacts used to support reported results.

## Setup

### Required files

The pipeline expects the following files to remain together:

```text
V14_Thesis_Pipeline.py
dataset_loader.py
configs/
```

`dataset_loader.py` supplies `prepare_dataset()` and
`_load_dataframe()`. At startup, V14 verifies these callables and
records the loader path, version, and SHA256 checksum.

### Python environment

A minimal CPU installation is:

```bash
pip install numpy pandas scipy scikit-learn optuna matplotlib psutil threadpoolctl
```

Packages for XGBoost, CatBoost, TabPFN, CodeCarbon, Parquet, and
interactive figures are installed only when the corresponding
experiment requires them:

```bash
pip install xgboost catboost tabpfn tabpfn-client codecarbon pyarrow plotly
```

### 1. Verify the appendix source

The following cell imports the appendix pipeline, checks that the
source parses as Python, and records its checksum. A thesis archive
should retain this checksum together with the exact JSON configuration
and environment manifest.

In [1]:
from pathlib import Path
import ast
import hashlib
import importlib.util
import json
import sys
import warnings

import pandas as pd

warnings.filterwarnings(
    "ignore",
    message="IProgress not found.*",
)

workspace = Path.cwd().resolve()
pipeline_path = workspace / "V14_Thesis_Pipeline.py"
loader_path = workspace / "dataset_loader.py"

if not pipeline_path.is_file():
    raise FileNotFoundError(f"Pipeline not found: {pipeline_path}")
if not loader_path.is_file():
    raise FileNotFoundError(f"Dataset loader not found: {loader_path}")

ast.parse(pipeline_path.read_text(encoding="utf-8"))
specification = importlib.util.spec_from_file_location(
    "v14_thesis_pipeline", pipeline_path
)
pipeline = importlib.util.module_from_spec(specification)
specification.loader.exec_module(pipeline)

source_identity = pd.DataFrame(
    [
        {
            "File": pipeline_path.name,
            "Pipeline version": pipeline.PIPELINE_VERSION,
            "Artifact schema": pipeline.ARTIFACT_SCHEMA_VERSION,
            "SHA256": hashlib.sha256(pipeline_path.read_bytes()).hexdigest(),
        },
        {
            "File": loader_path.name,
            "Pipeline version": "companion module",
            "Artifact schema": "not applicable",
            "SHA256": hashlib.sha256(loader_path.read_bytes()).hexdigest(),
        },
    ]
)
source_identity

,File,Pipeline version,Artifact schema,SHA256
0,V14_Thesis_Pipeline.py,V14,1.0,7214b28ceff1f2b9b365adce97d2cc968122837d68ff80...
1,dataset_loader.py,companion module,not applicable,2a0a4da97595f78c74547535132bcbeb99d3095c5b25e3...


## Steps

### 2. Follow the experiment sequence

Each run follows the same ordered sequence:

1. resolve JSON inheritance and defaults;
2. import and fingerprint the dataset loader;
3. load the dataset and verify target, groups, and timestamps;
4. resolve scenarios, sample sizes, models, devices, and CPU policy;
5. create the experiment matrix;
6. construct the outer test split and inner validation split;
7. fit preprocessing on training data only;
8. execute models sequentially at the Monte Carlo level;
9. persist splits, trials, predictions, metrics, runtime, and energy;
10. aggregate iteration-level evidence and generate figures; and
11. validate completion markers, checksums, metrics, and model sets.

The outer test partition is not used for hyperparameter selection.
Grouped splitting is applied when patient or subject identifiers are
supplied. Temporal splitting is available for ordered observations.

### 3. Understand the configuration hierarchy

A configuration may inherit from a base file through `extends`.
Nested JSON objects are merged; arrays and scalar values replace their
parent values. The resolved configuration is saved with the run.

| JSON section | Function |
|---|---|
| `data` | Source, outcome, grouping variable, exclusions |
| `splitting` | Stratified, grouped, or temporal partition rules |
| `preprocessing` | Train-only imputation and encoding |
| `sample_sizes`, `iterations`, `base_seed` | Experiment dimensions |
| `models` | Enablement, devices, search spaces, fixed parameters |
| `scenarios` | Budget reference and participating models |
| `cpu_parallelism`, `cpu_monitoring` | Resource policy and evidence |
| `energy` | Measurement scope and availability policy |
| `outputs`, `plots` | Storage and figure settings |
| `execution` | Error-handling and determinism policy |

### 4. Resolve an existing example

This cell resolves the tuned-ML-reference example without fitting a
model. It demonstrates inheritance and provides a compact audit of the
resulting scenario.

In [2]:
example_path = workspace / "configs" / "scenario_xgboost_budget.example.json"
example_raw = pipeline.v14_load_config_with_extends(example_path)
example_raw["experiment_name"] = "V14_Reader_Demonstration"
example_raw.setdefault("outputs", {})["root"] = "./V14_Experiments"
resolved_example = pipeline.v14_normalize_config(example_raw)

scenario_rows = []
for scenario_name in resolved_example["scenarios"]:
    scenario = pipeline.v14_resolve_scenario(
        resolved_example, scenario_name
    )
    reference_model = scenario.get("budget_reference_model")
    reference_is_tuned = bool(
        scenario["resolved_models"]
        .get(reference_model, {})
        .get("tuned_by_optuna", False)
    )
    scenario_rows.append(
        {
            "Scenario": scenario_name,
            "Models": ", ".join(scenario["enabled_models"]),
            "Reference model": reference_model,
            "Budgeting enabled": scenario["budgeting"]["enabled"],
            "Reference basis": (
                "reference_tuning_runtime"
                if reference_is_tuned
                else "reference_execution_runtime"
            ),
        }
    )

pd.DataFrame(scenario_rows)

,Scenario,Models,Reference model,Budgeting enabled,Reference basis
0,Custom_XGB_Runtime_Budget_Name,"TabPFN, L-SLR, Augmented_SLR, RandomForest, XG...",XGBoost,True,reference_tuning_runtime


### 5. Specify the study data

For the thesis dataset, replace every placeholder below with the
archived analysis definition. The grouping variable should identify
the independent unit (for example, a patient) whenever repeated
observations exist. Variables that reveal future information or encode
the outcome must be excluded before fitting.

In [3]:
thesis_data_definition = {
    "data": {
        "source": "PATH/TO/deidentified_analysis_dataset.parquet",
        "dataset_identifier": "archived_dataset_version",
        "target": "binary_outcome",
        "positive_label": 1,
        "group_column": "patient_id",
        "drop_columns": [
            "record_identifier",
            "post_outcome_variable",
        ],
        "keep_dataframe": True,
    },
    "splitting": {
        "strategy": "stratified_group",
        "group_aware": True,
        "require_groups": True,
        "strict": True,
        "candidate_splits": 256,
        "min_class_count_per_partition": 2,
    },
    "preprocessing": {
        "mode": "auto",
        "numeric_imputation": "median",
        "categorical_imputation_value": "__MISSING__",
        "categorical_encoding": "onehot",
    },
}

print(json.dumps(thesis_data_definition, indent=2))

{
  "data": {
    "source": "PATH/TO/deidentified_analysis_dataset.parquet",
    "dataset_identifier": "archived_dataset_version",
    "target": "binary_outcome",
    "positive_label": 1,
    "group_column": "patient_id",
    "drop_columns": [
      "record_identifier",
      "post_outcome_variable"
    ],
    "keep_dataframe": true
  },
  "splitting": {
    "strategy": "stratified_group",
    "group_aware": true,
    "require_groups": true,
    "strict": true,
    "candidate_splits": 256,
    "min_class_count_per_partition": 2
  },
  "preprocessing": {
    "mode": "auto",
    "numeric_imputation": "median",
    "categorical_imputation_value": "__MISSING__",
    "categorical_encoding": "onehot"
  }
}


### 6. Define models and scenarios

Model search spaces remain in JSON. Optuna uses a TPE sampler,
`NopPruner`, validation AUROC, and sequential trials (`n_jobs=1`).
Monte Carlo iterations are also sequential. Parallelism is confined to
the active model where supported by its library.

In [4]:
core_models = list(pipeline.MODEL_RUNNERS)
model_table = pd.DataFrame(
    [
        {
            "Model": model_name,
            "Enabled in example": bool(
                resolved_example["models"][model_name].get(
                    "enabled", False
                )
            ),
            "Optuna-tuned": bool(
                resolved_example["models"][model_name].get(
                    "tuned_by_optuna", False
                )
            ),
            "CPU parameter": pipeline.v14_cpu_model_thread_parameter(
                model_name
            ),
        }
        for model_name in core_models
    ]
)

assert set(core_models) == {
    "L-SLR",
    "Augmented_SLR",
    "RandomForest",
    "XGBoost",
    "CatBoost",
    "TabPFN",
}
model_table

,Model,Enabled in example,Optuna-tuned,CPU parameter
0,TabPFN,True,False,torch_and_native_threadpools
1,L-SLR,True,True,n_jobs_and_native_threadpools
2,Augmented_SLR,True,True,n_jobs_and_native_threadpools
3,RandomForest,True,True,n_jobs
4,XGBoost,True,True,n_jobs
5,CatBoost,True,True,thread_count


A scenario states which models are enabled and which model supplies the
reference runtime. Scenario names are labels rather than Python
constants. Sample sizes, iteration counts, context sizes, devices, and
trial limits are likewise configuration values.

Auxiliary comparators are recorded separately from the six core
models. They use the same test membership and labels as their paired
primary model, but they do not satisfy core-model completeness and are
excluded from ordinary core rankings.

### 7. Interpret runtime quantities

The runtime fields are intentionally non-interchangeable.

| Field | Interpretation |
|---|---|
| `Reference_Budget_Seconds` | Runtime assigned as the scientific reference |
| `Actual_Optuna_Tuning_Time_Seconds` | Dedicated HPO-loop stopwatch time |
| `Optuna_Tuning_Time_Capped_Seconds` | Eligible/capped competitor HPO time |
| `Final_Fit_Predict_Time_Seconds` | Post-selection refit and prediction |
| `Budget_Accounted_Runtime_Seconds` | Runtime counted by the defined budget accounting rule |
| `Actual_Total_Runtime_Seconds` | Broad observed model-execution wall-clock |

For a tuned reference such as XGBoost, the reference budget is the
dedicated Optuna tuning-loop wall-clock. Final refitting and test
prediction are timed separately and do not redefine that HPO budget.

For a non-tuned reference such as TabPFN, the configured
`reference_execution_runtime` basis is used because there is no
conventional per-dataset Optuna stage. This asymmetry is deliberate.

Wall-clock means elapsed real time measured as with a stopwatch; it is
not the sum of CPU time across cores.

### 8. Select the TabPFN context rule

The supported context strategies are:

- `fixed`: one deterministic context candidate;
- `adaptive`: measured growth and bracket refinement under the budget;
- `full`: the complete outer-training partition.

Context selection is based on training data. The full outer test
partition is retained for final evaluation.

In [5]:
context_examples = {
    "fixed": {
        "strategy": "fixed",
        "rows": 500,
    },
    "adaptive": {
        "strategy": "adaptive",
        "fraction": 0.20,
        "fraction_denominator": "outer_train",
    },
    "full": {
        "strategy": "full",
    },
}

context_examples

{'fixed': {'strategy': 'fixed', 'rows': 500},
 'adaptive': {'strategy': 'adaptive',
  'fraction': 0.2,
  'fraction_denominator': 'outer_train'},
 'full': {'strategy': 'full'}}

### 9. Save a configuration

A final study configuration should be stored as a standalone JSON file
under version control or in the thesis archive. The following pattern
inherits the shared defaults and changes only study-specific values:

```json
{
  "extends": "base_config.example.json",
  "enabled": true,
  "template_only": false,
  "experiment_name": "Final_Thesis_Experiment",
  "iterations": 15,
  "sample_sizes": [100, 300, "full"],
  "data": {
    "source": "PATH/TO/deidentified_analysis_dataset.parquet",
    "dataset_identifier": "archived_dataset_version",
    "target": "binary_outcome",
    "positive_label": 1,
    "group_column": "patient_id",
    "drop_columns": ["record_identifier"]
  },
  "splitting": {
    "strategy": "stratified_group",
    "require_groups": true,
    "strict": true
  },
  "outputs": {
    "root": "./V14_Experiments"
  }
}
```

The numerical values shown here illustrate syntax only. The archived
thesis configuration remains the authoritative definition.

### 10. Use the command-line interface

Run these commands from the directory containing the pipeline:

```powershell
# Built-in implementation checks
python V14_Thesis_Pipeline.py --self-test

# Resolve dependencies, data, devices, and the experiment matrix
python V14_Thesis_Pipeline.py --config path\to\experiment.json --dry-run

# Execute one configuration
python V14_Thesis_Pipeline.py --config path\to\experiment.json

# Execute enabled JSON files in a directory
python V14_Thesis_Pipeline.py --config-dir path\to\configuration_directory

# Resume a checksum-compatible run
python V14_Thesis_Pipeline.py --config path\to\experiment.json --resume --run-dir path\to\run

# Validate an existing run
python V14_Thesis_Pipeline.py --validate-run --run-dir path\to\run

# Rebuild figures from saved numerical artifacts
python V14_Thesis_Pipeline.py --regenerate-plots --run-dir path\to\run
```

A dry run should precede the definitive experiment. It imports the
loader, checks required packages and devices, resolves CPU policy, and
writes the planned experiment matrix without fitting models.

### 11. Read the artifact hierarchy

The principal evidence path is:

```text
exact split membership and predictions
    → iteration metrics
    → aggregated statistics
    → saved plot-data tables
    → figures
```

Each run includes resolved configurations, dataset and environment
manifests, split indices, model settings, Optuna trials, predictions,
runtime and energy evidence, completion markers, an artifact index, and
a validation report. Figures can therefore be regenerated without
rerunning the models.

Energy values must be interpreted according to their recorded scope.
In particular, client-side energy for a cloud TabPFN request does not
represent remote server energy.

## Checks

### 12. Run the lightweight implementation checks

These checks cover context semantics, comparator pairing and isolation,
runtime-budget definitions, sequential execution, CPU metadata, loader
compatibility, resume markers, and saved-data plot regeneration. They
do not fit the definitive thesis experiment.

In [6]:
self_test_report = pipeline.run_v14_self_tests()
assert self_test_report["status"] == "PASS"
pd.DataFrame(self_test_report["tests"])[
    ["test", "name", "status"]
]


V14 SELF TEST SUMMARY

Test 01: PASS - Fixed context semantics
Test 02: PASS - Adaptive context
Test 03: PASS - Full context
Test 04: PASS - Auxiliary comparator
Test 05: PASS - Comparator isolation
Test 06: PASS - Core ranking exclusion
Test 07: PASS - Paired comparison direction
Test 08: PASS - Tuned-reference budget definition
Test 09: PASS - Final-fit exclusion from HPO budget
Test 10: PASS - Non-tuned reference definition
Test 11: PASS - CPU maximum parallelism
Test 12: PASS - Optuna sequential
Test 13: PASS - Monte Carlo sequential
Test 14: PASS - CPU monitoring metadata
Test 15: PASS - dataset_loader dependency
Test 16: PASS - Resume validity
Test 17: PASS - Plot regeneration
Overall: PASS


,test,name,status
0,1,Fixed context semantics,PASS
1,2,Adaptive context,PASS
2,3,Full context,PASS
3,4,Auxiliary comparator,PASS
4,5,Comparator isolation,PASS
5,6,Core ranking exclusion,PASS
6,7,Paired comparison direction,PASS
7,8,Tuned-reference budget definition,PASS
8,9,Final-fit exclusion from HPO budget,PASS
9,10,Non-tuned reference definition,PASS


### 13. Pre-submission audit

Before reporting results, retain evidence that:

- the intended dataset version and predictor list were used;
- patient/group identifiers were supplied where required;
- the exact JSON configuration was archived;
- the environment and package versions were captured;
- all expected core models were recorded for every iteration;
- tuned-reference budgets equal the dedicated HPO duration;
- final fit and prediction remain separate from the HPO reference;
- comparator rows have exact paired split and label hashes;
- CPU and energy scopes are described accurately;
- `--validate-run` reports `PASS`; and
- figures regenerate from saved evidence.

A failed scientifically relevant validation should be resolved before
the run is treated as thesis evidence.

## Next steps

1. Replace the placeholder data definition with the archived study
   schema.
2. Review every enabled model, search space, device, sample size, and
   scenario with the supervisor.
3. Run `--dry-run` in the intended computing environment.
4. Execute a small preflight configuration before the definitive run.
5. Archive the final JSON, source checksums, environment manifest, and
   validation report with the thesis materials.
6. Describe software contributions and development provenance according
   to the university's research-integrity and disclosure requirements.

For appendix reading, the recommended order is: configuration,
experiment matrix, split audit, model runtime/prediction artifacts,
aggregate tables, and finally the validation report.